# 4.2. The Image Classification Dataset
D2L의 The Image Classification Dataset장을 PyTorch 기준으로 정리함.

이번 장에서는 Fashion-MNIST 이미지 데이터를 PyTorch로 불러오고,
모델이 학습할 수 있는 minibatch 형태로 만드는 방법을 공부한다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import time # 데이터 읽기 시간 측정용

import matplotlib.pyplot as plt # 이미지 출력용
import torch
from torch.utils.data import DataLoader # 데이터를 minibatch단위로 가져옴
from torchvision import datasets, transforms
# datasets = Fashion-MNIST같은 이미지 데이터 셋 제공
# transforms 이미지를 Tensor로 변환하거나 크기 변경

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. MNIST와 Fashion-MNIST

MNIST는 손으로 쓴 숫자 이미지를 분류하는 데이터셋이다.

예를 들어서 이런 이미지를 보고 숫자를 맞히는 문제다.

- 숫자 0 이미지 -> label 0
- 숫자 1 이미지 -> label 1
- 숫자 9 이미지 -> label 9

MNIST는 과거에 어려운 문제였지만, 현재는 간단한 모델도 높은 정확도를
낼 수 있어서 모델 성능을 제대로 비교하기에 너무 쉬운 데이터셋이 되었다.

그래서 D2L은 MNIST 대신 Fashion-MNIST를 사용한다.

Fashion-MNIST는 숫자가 아니고 옷 종류를 분류한다.

- 티셔츠
- 바지
- 스웨터
- 원피스
- 코트
- 샌들
- 셔츠
- 운동화
- 가방
- 앵클부츠

Fashion-MNIST는 10개 의류 범주로 구성되고, 원본 이미지는 28 × 28 크기의 흑백 이미지이다. 훈련 데이터는 60,000개, 테스트 데이터는 10,000개이다.

D2L에선 기존 MNIST보다 조금 더 현실적인 분류 연습용 데이터로 Fashion-MNIST를 사용한다.

| 구분 | 이미지 개수 | 용도 |
|---|---:|---|
| Training set | 60,000개 | 모델 학습 |
| Test set | 10,000개 | 학습이 끝난 모델 평가 |

각 class마다 다음과 같이 구성되어 있다.

- Training set: class당 6,000개
- Test set: class당 1,000개
- 전체 class 수: 10개

D2L 코드에는 테스트 데이터를 val이라는 이름으로 저장합니다. 하지만 Fashion-MNIST가 기본으로 제공하는 것은 train=True 훈련 split과 train=False 테스트 split이다.

엄밀한 실험이라면 훈련 데이터 일부를 validation set으로 따로 분리하고, test set은 최종 평가에만 사용하는 것이 맞다고 한다.

## 3. 이미지 변환 정의하기

Fashion-MNIST에서 가져온 원본 이미지는 바로 모델에 넣지 않고
먼저 PyTorch Tensor로 변환해야 한다.

`transforms.ToTensor()`는 이미지를 Tensor로 변환한다.

변환 전 이미지의 구조:

`[height, width]`

변환 후 이미지의 구조:

`[channel, height, width]`

Fashion-MNIST는 흑백 이미지이므로 channel 수가 1이다.

따라서 이미지 하나의 shape은 다음과 같다.

`[1, 28, 28]`

또한 픽셀값은 일반적으로 0~255 범위에서 0.0~1.0 범위의 float 값으로 변환된다.

이미지 한 장도 Tensor이다.
1 = channel 수, 28 = 이미지 높이, 28 = 이미지 너비

In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

## 4. Fashion-MNIST 불러오기

`datasets.FashionMNIST()`는 Fashion-MNIST 데이터셋을 내려받고
PyTorch에서 사용할 수 있도록 Dataset 객체로 만들어준다.

각 인자의 의미는 다음과 같다.

- `root="./data"`  
  데이터를 저장할 폴더이다.

- `train=True`  
  Training set 60,000개를 불러온다.

- `train=False`  
  Test set 10,000개를 불러온다.

- `transform=transform`  
  이미지를 가져올 때 앞에서 정의한 변환을 적용한다.

- `download=True`  
  데이터가 없다면 인터넷에서 내려받는다.

In [3]:
train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

100.0%
100.0%
100.0%
100.0%


In [4]:
print("훈련 데이터 개수:", len(train_dataset))
print("테스트 데이터 개수:", len(test_dataset))

훈련 데이터 개수: 60000
테스트 데이터 개수: 10000
